In [3]:
import torch
import torch.nn as nn

# 1. Model Mimarimizi OOP (Nesne Yönelimli) ve Temiz Mimariyle Tanımlıyoruz
class UAVAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(UAVAutoencoder, self).__init__()
        
        # ENCODER (Kodlayıcı - Sıkıştırma Katmanı)
        # Girişteki parametreleri (Örn: 12 kolonu) adım adım daraltıyoruz
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),                 # Aktivasyon Fonksiyonu
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8)           # Darboğaz (Bottleneck) - En öz rafine bilgi burada kalıyor
        )
        
        # DECODER (Çözücü - Yeniden Üretim Katmanı)
        # Sıkışan 8 boyutlu bilgiyi tekrar orijinal boyutuna genişletiyoruz
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)   # Çıkış boyutu girdi boyutuyla birebir aynı olmalı!
        )

    def forward(self, x):
        # Veri önce sıkışır, sonra tekrar açılır
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

# 2. Modelin Test Edilmesi (Prototipleme)
# Diyelim ki uçağımızın 12 adet parametresini (Sensör + Aktüatör) modele vereceğiz
ornek_girdi_boyutu = 12 
model = UAVAutoencoder(input_dim=ornek_girdi_boyutu)

print("====================================================================")
print("🤖 TERTEMİZ PYTORCH AUTOENCODER MİMARİSİ BAŞARIYLA OLUŞTURULDU 🤖")
print("====================================================================")
print(model)

🤖 TERTEMİZ PYTORCH AUTOENCODER MİMARİSİ BAŞARIYLA OLUŞTURULDU 🤖
UAVAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=12, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=8, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=12, bias=True)
  )
)


In [6]:
import os
import pandas as pd

# Masaüstündeki temiz verilerimizin yolu
clean_data_dir = r"C:\Users\feyza\Desktop\uav_project\data\processed\cleaned_train"

print("📂 Temiz uçuş klasörleri taranıyor ve tüm parametre haritası çıkarılıyor...\n")

# Dosya bazlı kolonları tutacağımız bir sözlük (dictionary)
file_columns_map = {}

# İlk 10 klasörü taramak veri yapısını anlamak için fazlasıyla yeterlidir
scanned_count = 0

for folder in os.listdir(clean_data_dir):
    if scanned_count >= 10:
        break
        
    folder_path = os.path.join(clean_data_dir, folder)
    if os.path.isdir(folder_path):
        scanned_count += 1
        files = os.listdir(folder_path)
        csv_files = [f for f in files if f.endswith('.csv')]
        
        for file_name in csv_files:
            # Dosya adındaki karmaşık sayıları temizleyip ana kategori adını alalım
            # Örn: "19_20_57_sensor_combined_0.csv" -> "sensor_combined"
            parts = file_name.split('_')
            if len(parts) >= 5:
                # Saat damgasından sonrasını birleştiriyoruz
                core_name = "_".join(parts[3:]).replace(".csv", "")
            else:
                core_name = file_name.replace(".csv", "")
                
            try:
                # Sadece ilk satırı okuyarak kolon isimlerini jet hızıyla çekiyoruz (RAM'i yormaz)
                df_temp = pd.read_csv(os.path.join(folder_path, file_name), nrows=1)
                cols = list(df_temp.columns)
                
                if core_name not in file_columns_map:
                    file_columns_map[core_name] = set(cols)
                else:
                    # Farklı uçuşlardaki ortak kolonları biriktiriyoruz
                    file_columns_map[core_name].update(cols)
            except Exception:
                continue

print("==========================================================================================")
print("📊 %100 TEMİZ VERİLERİNİZİN GERÇEK PARAMETRE KATALOĞU 📊")
print("==========================================================================================")

total_features = 0
for file_type, columns in file_columns_map.items():
    # timestamp kolonunu görsel kalabalık yapmasın diye listeden çıkarıp gösterelim
    clean_cols = [c for c in columns if c != 'timestamp']
    total_features += len(clean_cols)
    print(f"\n📂 Dosya Türü: {file_type}.csv (Toplam {len(clean_cols)} Kritik Kolon)")
    print(f"   ↳ Kolonlar: {sorted(clean_cols)}")
    print("-" * 90)

print(f"\n🚀 TARAMA TAMAMLANDI! Uçağımızda toplam {total_features} adet kullanılabilir benzersiz parametre bulundu.")

📂 Temiz uçuş klasörleri taranıyor ve tüm parametre haritası çıkarılıyor...

📊 %100 TEMİZ VERİLERİNİZİN GERÇEK PARAMETRE KATALOĞU 📊

📂 Dosya Türü: actuator_controls_0_0.csv (Toplam 9 Kritik Kolon)
   ↳ Kolonlar: ['control[0]', 'control[1]', 'control[2]', 'control[3]', 'control[4]', 'control[5]', 'control[6]', 'control[7]', 'timestamp_sample']
------------------------------------------------------------------------------------------

📂 Dosya Türü: actuator_outputs_0.csv (Toplam 17 Kritik Kolon)
   ↳ Kolonlar: ['noutputs', 'output[0]', 'output[10]', 'output[11]', 'output[12]', 'output[13]', 'output[14]', 'output[15]', 'output[1]', 'output[2]', 'output[3]', 'output[4]', 'output[5]', 'output[6]', 'output[7]', 'output[8]', 'output[9]']
------------------------------------------------------------------------------------------

📂 Dosya Türü: battery_status_0.csv (Toplam 25 Kritik Kolon)
   ↳ Kolonlar: ['average_current_a', 'average_time_to_empty', 'capacity', 'cell_count', 'connected', 'curren

In [7]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# Klasör Yolu
clean_data_dir = r"C:\Users\feyza\Desktop\uav_project\data\processed\cleaned_train"

# Kataloğumuzdan seçtiğimiz %100 garantili 30 kritik parametre
features_to_train = [
    'gyro_rad[0]', 'gyro_rad[1]', 'gyro_rad[2]',
    'accelerometer_m_s2[0]', 'accelerometer_m_s2[1]', 'accelerometer_m_s2[2]',
    'baro_alt_meter', 'baro_pressure_pa', 'baro_temp_celcius',
    'q[0]', 'q[1]', 'q[2]', 'q[3]',
    'output[0]', 'output[1]', 'output[2]', 'output[3]', 'output[4]', 'output[5]', 'output[6]', 'output[7]',
    'x', 'y', 'z', 'vx', 'vy', 'vz', 'ax', 'ay', 'az'
]

print("🔍 Adım 1: Temiz veriler tam eşleşmeyle yükleniyor...")
all_dfs = []
loaded_count = 0
max_flights = 15  # Kararlı bir başlangıç için 15 temiz uçuşu birleştiriyoruz

for folder in os.listdir(clean_data_dir):
    if loaded_count >= max_flights:
        break
        
    folder_path = os.path.join(clean_data_dir, folder)
    if os.path.isdir(folder_path):
        files = os.listdir(folder_path)
        
        # Nokta atışı birebir dosya tespiti (dhl_ önekli karmaşayı engelliyoruz)
        imu_file = [f for f in files if f.endswith("sensor_combined_0.csv")]
        air_file = [f for f in files if f.endswith("vehicle_air_data_0.csv")]
        att_file = [f for f in files if f.endswith("vehicle_attitude_0.csv")]
        act_file = [f for f in files if f.endswith("actuator_outputs_0.csv")]
        loc_file = [f for f in files if f.endswith("vehicle_local_position_0.csv")]
        
        if imu_file and air_file and att_file and act_file and loc_file:
            try:
                df_imu = pd.read_csv(os.path.join(folder_path, imu_file[0])).sort_values('timestamp')
                df_air = pd.read_csv(os.path.join(folder_path, air_file[0])).sort_values('timestamp')
                df_att = pd.read_csv(os.path.join(folder_path, att_file[0])).sort_values('timestamp')
                df_act = pd.read_csv(os.path.join(folder_path, act_file[0])).sort_values('timestamp')
                df_loc = pd.read_csv(os.path.join(folder_path, loc_file[0])).sort_values('timestamp')
                
                # Jilet gibi Zaman Serisi Hizalama Zinciri
                df_m = pd.merge_asof(df_imu, df_air, on='timestamp', direction='nearest')
                df_m = pd.merge_asof(df_m, df_att, on='timestamp', direction='nearest')
                df_m = pd.merge_asof(df_m, df_act, on='timestamp', direction='nearest')
                df_m = pd.merge_asof(df_m, df_loc, on='timestamp', direction='nearest')
                
                if all(col in df_m.columns for col in features_to_train):
                    all_dfs.append(df_m[features_to_train])
                    loaded_count += 1
                    print(f" 🟢 [{loaded_count}/{max_flights}] Dosya Evlendirildi -> {folder}.ulg")
            except Exception as e:
                continue

if len(all_dfs) > 0:
    df_master_clean = pd.concat(all_dfs, ignore_index=True).fillna(0)
    print(f"\n📊 VERİ TABANI HAZIR: {df_master_clean.shape[0]} satır x {df_master_clean.shape[1]} kolon")
    
    # Ölçeklendirme
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_master_clean)
    
    class UAVDataset(Dataset):
        def __init__(self, data):
            self.data = torch.tensor(data, dtype=torch.float32)
        def __len__(self):
            return len(self.data)
        def __getitem__(self, idx):
            return self.data[idx], self.data[idx]

    dataset = UAVDataset(scaled_data)
    dataloader = DataLoader(dataset, batch_size=512, shuffle=True)
    
    # 30 Kolona Özel Kusursuz Autoencoder Tasarımı
    input_dim = len(features_to_train)
    class UAVFinalAutoencoder(nn.Module):
        def __init__(self, dim):
            super(UAVFinalAutoencoder, self).__init__()
            self.encoder = nn.Sequential(
                nn.Linear(dim, 64),
                nn.ReLU(),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, 16) # Darboğaz Katmanı
            )
            self.decoder = nn.Sequential(
                nn.Linear(16, 32),
                nn.ReLU(),
                nn.Linear(32, 64),
                nn.ReLU(),
                nn.Linear(64, dim)
            )
        def forward(self, x):
            return self.decoder(self.encoder(x))

    model = UAVFinalAutoencoder(dim=input_dim)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    epochs = 10
    print("\n🚀 YAPAY ZEKÂ MODEL EĞİTİMİ CANLI OLARAK BAŞLADI...\n")
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * inputs.size(0)
            
        total_epoch_loss = epoch_loss / len(dataloader.dataset)
        print(f"📈 Tur [{epoch+1}/{epochs}] -> Ortalama Hata (Loss): {total_epoch_loss:.6f}")
        
    print("\n🏆 MÜKEMMEL! Yapay zeka uçağın 30 kritik organını hatasız ezberledi.")
else:
    print("❌ Hata: Seçilen kombinasyona uyan temiz veri bulunamadı.")

🔍 Adım 1: Temiz veriler tam eşleşmeyle yükleniyor...
 🟢 [1/15] Dosya Evlendirildi -> 00_02_49.ulg
 🟢 [2/15] Dosya Evlendirildi -> 00_05_28.ulg
 🟢 [3/15] Dosya Evlendirildi -> 00_05_29.ulg
 🟢 [4/15] Dosya Evlendirildi -> 00_09_34.ulg
 🟢 [5/15] Dosya Evlendirildi -> 00_15_34.ulg
 🟢 [6/15] Dosya Evlendirildi -> 00_44_00.ulg
 🟢 [7/15] Dosya Evlendirildi -> 01_48_01.ulg
 🟢 [8/15] Dosya Evlendirildi -> 01_55_04.ulg
 🟢 [9/15] Dosya Evlendirildi -> 02_32_21.ulg
 🟢 [10/15] Dosya Evlendirildi -> 02_39_00.ulg
 🟢 [11/15] Dosya Evlendirildi -> 03_16_07.ulg
 🟢 [12/15] Dosya Evlendirildi -> 03_44_01.ulg
 🟢 [13/15] Dosya Evlendirildi -> 07_21_02.ulg
 🟢 [14/15] Dosya Evlendirildi -> 07_23_56.ulg
 🟢 [15/15] Dosya Evlendirildi -> 07_24_27.ulg

📊 VERİ TABANI HAZIR: 604737 satır x 30 kolon

🚀 YAPAY ZEKÂ MODEL EĞİTİMİ CANLI OLARAK BAŞLADI...

📈 Tur [1/10] -> Ortalama Hata (Loss): 0.227782
📈 Tur [2/10] -> Ortalama Hata (Loss): 0.066170
📈 Tur [3/10] -> Ortalama Hata (Loss): 0.042332
📈 Tur [4/10] -> Ortalama H

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# ==========================================================================================
# 1. AYARLAR VE PARAMETRELER
# ==========================================================================================
clean_data_dir = r"C:\Users\feyza\Desktop\uav_project\data\processed\cleaned_train"

features_to_train = [
    'gyro_rad[0]', 'gyro_rad[1]', 'gyro_rad[2]',
    'accelerometer_m_s2[0]', 'accelerometer_m_s2[1]', 'accelerometer_m_s2[2]',
    'baro_alt_meter', 'baro_pressure_pa', 'baro_temp_celcius',
    'q[0]', 'q[1]', 'q[2]', 'q[3]',
    'output[0]', 'output[1]', 'output[2]', 'output[3]', 'output[4]', 'output[5]', 'output[6]', 'output[7]',
    'x', 'y', 'z', 'vx', 'vy', 'vz', 'ax', 'ay', 'az'
]

print("🔍 Adım 1: Temiz veriler tam eslesmeyle yukleniyor...")
all_dfs = []
loaded_count = 0
max_flights = 15 

for folder in os.listdir(clean_data_dir):
    if loaded_count >= max_flights:
        break
    folder_path = os.path.join(clean_data_dir, folder)
    if os.path.isdir(folder_path):
        files = os.listdir(folder_path)
        
        imu_file = [f for f in files if f.endswith("sensor_combined_0.csv")]
        air_file = [f for f in files if f.endswith("vehicle_air_data_0.csv")]
        att_file = [f for f in files if f.endswith("vehicle_attitude_0.csv")]
        act_file = [f for f in files if f.endswith("actuator_outputs_0.csv")]
        loc_file = [f for f in files if f.endswith("vehicle_local_position_0.csv")]
        
        if imu_file and air_file and att_file and act_file and loc_file:
            try:
                df_imu = pd.read_csv(os.path.join(folder_path, imu_file[0])).sort_values('timestamp')
                df_air = pd.read_csv(os.path.join(folder_path, air_file[0])).sort_values('timestamp')
                df_att = pd.read_csv(os.path.join(folder_path, att_file[0])).sort_values('timestamp')
                df_act = pd.read_csv(os.path.join(folder_path, act_file[0])).sort_values('timestamp')
                df_loc = pd.read_csv(os.path.join(folder_path, loc_file[0])).sort_values('timestamp')
                
                df_m = pd.merge_asof(df_imu, df_air, on='timestamp', direction='nearest')
                df_m = pd.merge_asof(df_m, df_att, on='timestamp', direction='nearest')
                df_m = pd.merge_asof(df_m, df_act, on='timestamp', direction='nearest')
                df_m = pd.merge_asof(df_m, df_loc, on='timestamp', direction='nearest')
                
                if all(col in df_m.columns for col in features_to_train):
                    all_dfs.append(df_m[features_to_train])
                    loaded_count += 1
                    print(f" 🟢 [{loaded_count}/{max_flights}] Dosya Yuklendi -> {folder}.ulg")
            except Exception:
                continue

if len(all_dfs) > 0:
    df_master_clean = pd.concat(all_dfs, ignore_index=True).fillna(0)
    print(f"\n📊 VERİ TABANI HAZIR: {df_master_clean.shape[0]} satır x {df_master_clean.shape[1]} kolon")
    
    # Ölçeklendirme
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_master_clean)
    
    class UAVDataset(Dataset):
        def __init__(self, data):
            self.data = torch.tensor(data, dtype=torch.float32)
        def __len__(self):
            return len(self.data)
        def __getitem__(self, idx):
            return self.data[idx], self.data[idx]

    dataset = UAVDataset(scaled_data)
    dataloader = DataLoader(dataset, batch_size=512, shuffle=True)
    
    # ==========================================================================================
    # 2. MODEL MİMARİSİ VE PARAMETRELER
    # ==========================================================================================
    input_dim = len(features_to_train)
    class UAVFinalAutoencoder(nn.Module):
        def __init__(self, dim):
            super(UAVFinalAutoencoder, self).__init__()
            self.encoder = nn.Sequential(
                nn.Linear(dim, 64),
                nn.ReLU(),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, 16)
            )
            self.decoder = nn.Sequential(
                nn.Linear(16, 32),
                nn.ReLU(),
                nn.Linear(32, 64),
                nn.ReLU(),
                nn.Linear(64, dim)
            )
        def forward(self, x):
            return self.decoder(self.encoder(x))

    model = UAVFinalAutoencoder(dim=input_dim)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # ==========================================================================================
    # 3. EĞİTİM DÖNGÜSÜ
    # ==========================================================================================
    epochs = 10
    print("\n🚀 YAPAY ZEKÂ MODEL EĞİTİMİ BAŞLADI...\n")
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * inputs.size(0)
            
        total_epoch_loss = epoch_loss / len(dataloader.dataset)
        print(f"📈 Tur [{epoch+1}/{epochs}] -> Loss: {total_epoch_loss:.6f}")
        
    print("\n🏆 MODEL BAŞARIYLA EĞİTİLDİ!")
    
    # ==========================================================================================
    # 4. OTOMATİK DİSKE MÜHÜRLERME (KAYDETME)
    # ==========================================================================================
    model_save_dir = r"C:\Users\feyza\Desktop\uav_project\models"
    os.makedirs(model_save_dir, exist_ok=True)
    model_path = os.path.join(model_save_dir, "uav_autoencoder_v1.pth")
    
    torch.save(model.state_dict(), model_path)
    print("\n==========================================================================================")
    print(f"💾 MUTLU SON! Yapay zeka beyni ölümsüzleştirildi.")
    print(f"📍 Dosya Konumu: {model_path}")
    print("==========================================================================================")
else:
    print("❌ Hata: Veri bulunamadi.")

🔍 Adım 1: Temiz veriler tam eslesmeyle yukleniyor...
 🟢 [1/15] Dosya Yuklendi -> 00_02_49.ulg
 🟢 [2/15] Dosya Yuklendi -> 00_05_28.ulg
 🟢 [3/15] Dosya Yuklendi -> 00_05_29.ulg
 🟢 [4/15] Dosya Yuklendi -> 00_09_34.ulg
 🟢 [5/15] Dosya Yuklendi -> 00_15_34.ulg
 🟢 [6/15] Dosya Yuklendi -> 00_44_00.ulg
 🟢 [7/15] Dosya Yuklendi -> 01_48_01.ulg
 🟢 [8/15] Dosya Yuklendi -> 01_55_04.ulg
 🟢 [9/15] Dosya Yuklendi -> 02_32_21.ulg
 🟢 [10/15] Dosya Yuklendi -> 02_39_00.ulg
 🟢 [11/15] Dosya Yuklendi -> 03_16_07.ulg
 🟢 [12/15] Dosya Yuklendi -> 03_44_01.ulg
 🟢 [13/15] Dosya Yuklendi -> 07_21_02.ulg
 🟢 [14/15] Dosya Yuklendi -> 07_23_56.ulg
 🟢 [15/15] Dosya Yuklendi -> 07_24_27.ulg

📊 VERİ TABANI HAZIR: 604737 satır x 30 kolon

🚀 YAPAY ZEKÂ MODEL EĞİTİMİ BAŞLADI...

📈 Tur [1/10] -> Loss: 0.230445
📈 Tur [2/10] -> Loss: 0.072184
📈 Tur [3/10] -> Loss: 0.048180
📈 Tur [4/10] -> Loss: 0.038635
📈 Tur [5/10] -> Loss: 0.032801
📈 Tur [6/10] -> Loss: 0.028157
📈 Tur [7/10] -> Loss: 0.024611
📈 Tur [8/10] -> Loss: 0

In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# ==========================================================================================
# 1. AYARLAR VE YENİ VERİ PIPELINE'I
# ==========================================================================================
clean_data_dir = r"C:\Users\feyza\Desktop\uav_project\data\processed\cleaned_train"
model_path = r"C:\Users\feyza\Desktop\uav_project\models\uav_autoencoder_v1.pth"

features_to_train = [
    'gyro_rad[0]', 'gyro_rad[1]', 'gyro_rad[2]',
    'accelerometer_m_s2[0]', 'accelerometer_m_s2[1]', 'accelerometer_m_s2[2]',
    'baro_alt_meter', 'baro_pressure_pa', 'baro_temp_celcius',
    'q[0]', 'q[1]', 'q[2]', 'q[3]',
    'output[0]', 'output[1]', 'output[2]', 'output[3]', 'output[4]', 'output[5]', 'output[6]', 'output[7]',
    'x', 'y', 'z', 'vx', 'vy', 'vz', 'ax', 'ay', 'az'
]

print("🔍 Adım 1: İlk 15 uçuş pas geçiliyor, yeni 30 temiz uçuş yükleniyor...")
all_dfs = []
skipped_count = 0
loaded_count = 0

# İlk eğittiğimiz 15 uçuşu atlayıp, sonraki 30 uçuşu almak için sınırları koyuyoruz
max_skip = 15
max_new_flights = 30

for folder in os.listdir(clean_data_dir):
    folder_path = os.path.join(clean_data_dir, folder)
    if os.path.isdir(folder_path):
        files = os.listdir(folder_path)
        
        imu_file = [f for f in files if f.endswith("sensor_combined_0.csv")]
        air_file = [f for f in files if f.endswith("vehicle_air_data_0.csv")]
        att_file = [f for f in files if f.endswith("vehicle_attitude_0.csv")]
        act_file = [f for f in files if f.endswith("actuator_outputs_0.csv")]
        loc_file = [f for f in files if f.endswith("vehicle_local_position_0.csv")]
        
        if imu_file and air_file and att_file and act_file and loc_file:
            # İlk 15 kararlı dosyayı atla (onları zaten öğrendi)
            if skipped_count < max_skip:
                skipped_count += 1
                continue
                
            if loaded_count >= max_new_flights:
                break
                
            try:
                df_imu = pd.read_csv(os.path.join(folder_path, imu_file[0])).sort_values('timestamp')
                df_air = pd.read_csv(os.path.join(folder_path, air_file[0])).sort_values('timestamp')
                df_att = pd.read_csv(os.path.join(folder_path, att_file[0])).sort_values('timestamp')
                df_act = pd.read_csv(os.path.join(folder_path, act_file[0])).sort_values('timestamp')
                df_loc = pd.read_csv(os.path.join(folder_path, loc_file[0])).sort_values('timestamp')
                
                df_m = pd.merge_asof(df_imu, df_air, on='timestamp', direction='nearest')
                df_m = pd.merge_asof(df_m, df_att, on='timestamp', direction='nearest')
                df_m = pd.merge_asof(df_m, df_act, on='timestamp', direction='nearest')
                df_m = pd.merge_asof(df_m, df_loc, on='timestamp', direction='nearest')
                
                if all(col in df_m.columns for col in features_to_train):
                    all_dfs.append(df_m[features_to_train])
                    loaded_count += 1
                    print(f" 🟢 EXTRA [{loaded_count}/{max_new_flights}] Yeni Uçuş Eklendi -> {folder}.ulg")
            except Exception:
                continue

if len(all_dfs) > 0:
    df_master_clean = pd.concat(all_dfs, ignore_index=True).fillna(0)
    print(f"\n📊 YENİ EK VERİ DETAYI: {df_master_clean.shape[0]} satır ekleniyor...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_master_clean)
    
    class UAVDataset(Dataset):
        def __init__(self, data):
            self.data = torch.tensor(data, dtype=torch.float32)
        def __len__(self):
            return len(self.data)
        def __getitem__(self, idx):
            return self.data[idx], self.data[idx]

    dataset = UAVDataset(scaled_data)
    dataloader = DataLoader(dataset, batch_size=512, shuffle=True)
    
    # ==========================================================================================
    # 2. MODELİN TANIMLANMASI VE ESKİ HAFIZANIN ENJEKTE EDİLMESİ
    # ==========================================================================================
    input_dim = len(features_to_train)
    class UAVFinalAutoencoder(nn.Module):
        def __init__(self, dim):
            super(UAVFinalAutoencoder, self).__init__()
            self.encoder = nn.Sequential(
                nn.Linear(dim, 64), nn.ReLU(),
                nn.Linear(64, 32), nn.ReLU(),
                nn.Linear(32, 16)
            )
            self.decoder = nn.Sequential(
                nn.Linear(16, 32), nn.ReLU(),
                nn.Linear(32, 64), nn.ReLU(),
                nn.Linear(64, dim)
            )
        def forward(self, x):
            return self.decoder(self.encoder(x))

    model = UAVFinalAutoencoder(dim=input_dim)
    
    # 📍 İŞTE SİHİRLİ DOKUNUŞ: Diskteki eski 41 KB'lık beyni okuyup modelin içine yüklüyoruz
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path))
        print("🔌 Eski model ağırlıkları başarıyla yüklendi! Eğitim sıfırdan değil, kaldığı yerden devam ediyor.")
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005) # İnce ayar (Fine-tuning) için LR'ı biraz düşürdük ki eski bilgileri sarsmasın
    
    # ==========================================================================================
    # 3. İNCE AYAR (FINE-TUNING) EĞİTİMİ
    # ==========================================================================================
    epochs = 7 # Yeni verileri sindirmesi için 7 tur kafi
    print("\n🚀 MODELİN HASSASİYET EĞİTİMİ BAŞLADI...\n")
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * inputs.size(0)
            
        total_epoch_loss = epoch_loss / len(dataloader.dataset)
        print(f"📈 Ek Hassasiyet Turu [{epoch+1}/{epochs}] -> İnce Ayar Loss: {total_epoch_loss:.6f}")
        
    print("\n🏆 SÜPER HASSAS MODEL BAŞARIYLA EĞİTİLDİ!")
    
    # ==========================================================================================
    # 4. GÜNCEL MODELİ YENİDEN MÜHÜRLERME
    # ==========================================================================================
    torch.save(model.state_dict(), model_path)
    print("\n==========================================================================================")
    print(f"💾 ZAFER! Üst düzey hassas yapay zeka beyni diskte güncellendi.")
    print(f"📍 Dosya Konumu: {model_path}")
    print("==========================================================================================")
else:
    print("❌ Hata: İşlenecek yeni temiz veri bulunamadı.")

🔍 Adım 1: İlk 15 uçuş pas geçiliyor, yeni 30 temiz uçuş yükleniyor...
 🟢 EXTRA [1/30] Yeni Uçuş Eklendi -> 07_25_53.ulg
 🟢 EXTRA [2/30] Yeni Uçuş Eklendi -> 07_26_23.ulg
 🟢 EXTRA [3/30] Yeni Uçuş Eklendi -> 07_29_03.ulg
 🟢 EXTRA [4/30] Yeni Uçuş Eklendi -> 07_29_31.ulg
 🟢 EXTRA [5/30] Yeni Uçuş Eklendi -> 07_33_39.ulg
 🟢 EXTRA [6/30] Yeni Uçuş Eklendi -> 07_34_37.ulg
 🟢 EXTRA [7/30] Yeni Uçuş Eklendi -> 07_36_08.ulg
 🟢 EXTRA [8/30] Yeni Uçuş Eklendi -> 07_38_12.ulg
 🟢 EXTRA [9/30] Yeni Uçuş Eklendi -> 07_45_20.ulg
 🟢 EXTRA [10/30] Yeni Uçuş Eklendi -> 07_45_29.ulg
 🟢 EXTRA [11/30] Yeni Uçuş Eklendi -> 07_49_17.ulg
 🟢 EXTRA [12/30] Yeni Uçuş Eklendi -> 07_58_08.ulg
 🟢 EXTRA [13/30] Yeni Uçuş Eklendi -> 08_44_40.ulg
 🟢 EXTRA [14/30] Yeni Uçuş Eklendi -> 12_34_09.ulg
 🟢 EXTRA [15/30] Yeni Uçuş Eklendi -> 13_19_20.ulg
 🟢 EXTRA [16/30] Yeni Uçuş Eklendi -> 14_06_20.ulg
 🟢 EXTRA [17/30] Yeni Uçuş Eklendi -> 14_39_39.ulg
 🟢 EXTRA [18/30] Yeni Uçuş Eklendi -> 16_57_53.ulg
 🟢 EXTRA [19/30] Yeni